# 01 · fMRI temporal interpolation — results

**My part of the project.** A 3D U-Net predicts the missing middle BOLD volume from its two temporal neighbours:

$$\hat V_{t+1} = f(V_t,\; V_{t+2})$$

Applied across a run this **doubles the effective temporal resolution**. This notebook reproduces the midterm-style results:

1. **Tri-planar** Ground-truth vs Prediction vs `|Error|` (axial / coronal / sagittal)
2. **Error localization** — MIP projections + worst-slice anatomy overlay
3. **Naive baseline** comparison ( $0.5(V_t+V_{t+2})$ ) — does the model actually help?
4. **Metrics across the run** (L1 / PSNR) + training loss curve

All plotting helpers are in [`nb_utils.py`](nb_utils.py).

In [ ]:
%load_ext autoreload
%autoreload 2
import nb_utils as nb

cfg = nb.Config(data_dir="/srv/fMRI-data", bold_file=None,
                norm_mode="zscore", residual=False)
bold = cfg.resolve_bold()
print("run:", bold.name)
nb.bold_info(bold)

## Load the model

Pretrained checkpoint: `data_interpolation/checkpoints/pretrained/model_weights.pt` (zscore norm, non-residual).

In [ ]:
interp = nb.load_interpolator(cfg)
print("device:", interp.device)

## Pick a timepoint and predict

`predict_triplet` reuses the package's dataset so the normalization matches training/eval exactly, then denormalizes everything back to physical BOLD units.

In [ ]:
T_INDEX = 130   # the held-out frame to reconstruct (predict V_{t+1} from V_t, V_{t+2})
res = nb.predict_triplet(cfg, interp, bold, t=T_INDEX)
nb.triplet_metrics(res)

## 1 · Tri-planar — Ground truth vs Prediction vs |Error|

The headline figure: how close is the reconstructed frame, and where does it differ?

In [ ]:
fig = nb.triplanar_compare(res, save_path=cfg.out_dir / f"triplanar_t{T_INDEX}.png")
fig

## 2 · Error localization

Maximum-intensity projections of the `|error|` volume (top-down / front / side), plus the single worst axial slice with the error overlaid on the anatomy. This shows whether errors cluster at edges / vessels / high-motion regions rather than being uniform.

In [ ]:
fig = nb.error_localization(res, save_path=cfg.out_dir / f"error_loc_t{T_INDEX}.png")
fig

## 3 · Does the model beat the naive midpoint baseline?

The naive baseline simply averages the neighbours, $0.5(V_t+V_{t+2})$. A useful model must do better than that. Here we compare the two error maps side by side at the worst slice.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

err_m, err_n = res.error, res.naive_error
z = int(err_n.sum(axis=(0, 1)).argmax())
emax = float(np.percentile(err_n, 99.0)) or 1.0
vmax = float(np.percentile(res.gt, 99.5)) or 1.0
m = nb.triplet_metrics(res)

fig, ax = plt.subplots(1, 4, figsize=(18, 5))
panels = [(res.gt[:, :, z].T, "gray", (0, vmax), f"Ground truth (z={z})"),
          (res.pred[:, :, z].T, "gray", (0, vmax), "Model prediction"),
          (err_n[:, :, z].T, "jet", (0, emax), f"|Error| naive\nL1={m['naive_l1']:.4f}"),
          (err_m[:, :, z].T, "jet", (0, emax), f"|Error| model\nL1={m['model_l1']:.4f}")]
for a, (data, cmap, lim, title) in zip(ax, panels):
    im = a.imshow(data, origin="lower", cmap=cmap, vmin=lim[0], vmax=lim[1])
    a.set_title(title, fontsize=11); a.axis("off")
    if cmap == "jet":
        fig.colorbar(im, ax=a, fraction=0.046, pad=0.04)
fig.suptitle("Model vs naive baseline — same slice, same error scale", fontsize=13)
fig.tight_layout()
fig.savefig(cfg.out_dir / f"model_vs_naive_t{T_INDEX}.png", dpi=120, bbox_inches="tight")
plt.show()

## 4 · Metrics across the whole run

Sweep every valid `t` (use a stride to keep it fast) and aggregate. We report L1 and PSNR for the model and the naive baseline, and the fraction of frames where the model wins.

In [ ]:
import pandas as pd

rows = nb.sweep_metrics(cfg, interp, bold, stride=10)   # stride=1 for the full sweep
df = pd.DataFrame(rows)
summary = pd.DataFrame({
    "mean_L1":   [df.model_l1.mean(),   df.naive_l1.mean()],
    "mean_PSNR": [df.model_psnr.mean(), df.naive_psnr.mean()],
}, index=["model", "naive"])
win = float((df.model_l1 < df.naive_l1).mean() * 100)
print(f"model beats naive on {win:.1f}% of frames (n={len(df)})")
summary

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(df.t, df.model_l1, label="model", lw=2)
ax[0].plot(df.t, df.naive_l1, label="naive", lw=2, ls="--")
ax[0].set_xlabel("t"); ax[0].set_ylabel("L1 (normalized)"); ax[0].set_title("L1 per frame")
ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(df.t, df.model_psnr, label="model", lw=2)
ax[1].plot(df.t, df.naive_psnr, label="naive", lw=2, ls="--")
ax[1].set_xlabel("t"); ax[1].set_ylabel("PSNR (dB)"); ax[1].set_title("PSNR per frame")
ax[1].legend(); ax[1].grid(alpha=0.3)
fig.tight_layout()
fig.savefig(cfg.out_dir / "metrics_curves.png", dpi=120)
plt.show()

## Training loss curve

In [ ]:
if cfg.interp_history.is_file():
    nb.show_loss_curve(cfg.interp_history, save_path=cfg.out_dir / "loss_curve.png")
else:
    print("no history.json at", cfg.interp_history)

## (Optional) Write a 2× interpolated NIfTI

Generate the actual upsampled run — original frames plus a synthetic frame between each pair (`2T-1` frames, TR halved). This is the artifact the rest of the pipeline can consume.

In [ ]:
# Uncomment to run (writes a full 4D NIfTI — can take a minute per ~150 frames):
# out_nii = cfg.out_dir / (bold.stem.replace('.nii','') + '_2x.nii.gz')
# interp.interpolate(str(bold), str(out_nii), mode='insert')
# print('wrote', out_nii)

**Takeaways for the slide:**
- The model reconstructs $V_{t+1}$ from neighbours with low residual; errors concentrate at edges / high-frequency structure (see the localization panel).
- It beats the naive midpoint average on the majority of frames in both L1 and PSNR.
- Next: feed the interpolated run into the full restoration pipeline → **`02_pipeline_comparison.ipynb`**.